In [1]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

import tensorflow as tf

from PIL import Image

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score, roc_auc_score, classification_report

from keras.utils import Sequence, to_categorical
from keras.optimizers import Adam
from keras import optimizers, metrics, layers, models, applications
# from keras.callbacks import ReduceLROnPlateau
from keras.models import load_model, Model, Sequential
# from keras.preprocessing.image import ImageDataGenerator
from keras.layers import Input, Conv2D, MaxPooling2D, UpSampling2D, Flatten, Dense, GlobalAveragePooling2D, PReLU, Dropout, BatchNormalization
from keras.applications import ResNet50

In [2]:
# folder_path = r"D:\GSoC\DeepLense\dataset"
folder_path = r"../dataset"
types = ['no', 'sphere', 'vort']

def load_data(split):
    data = []
    labels = []
    
    for t in types:
        path = os.path.join(folder_path, split, t)
        if t == 'no':
            label = 0
        elif t == 'sphere':
            label = 1
        else:
            label = 2
        for npy_name in os.listdir(path):
            npy_path = os.path.join(path, npy_name)
            arr = np.load(npy_path)
            data.append(arr)
            labels.append(label)
    
    return np.array(data), np.array(labels)

In [3]:
X_train, Y_train = load_data('train')
X_val, Y_val = load_data('val')

X_train = X_train.transpose(0, 2, 3, 1)
X_val = X_val.transpose(0, 2, 3, 1)

print(f"Train shape: {X_train.shape}, Labels: {Y_train.shape}")
print(f"Validation shape: {X_val.shape}, Labels: {Y_val.shape}")

Y_train_cat = to_categorical(Y_train, num_classes=3)
Y_val_cat = to_categorical(Y_val, num_classes=3)

Train shape: (30000, 150, 150, 1), Labels: (30000,)
Validation shape: (7500, 150, 150, 1), Labels: (7500,)


In [4]:
input_shape = (150, 150, 1)
width = 128
num_epochs = 30 
batch_size = 128
lr = 1e-4

In [16]:
def encoder():
    resnet = applications.ResNet50(weights=None, include_top=False, input_shape=input_shape)
    model = models.Sequential([
        resnet, 
            # first layer
            layers.GlobalAveragePooling2D(), 
            layers.Dense(width*8,  activation='relu'),
            layers.BatchNormalization(), 
            layers.Dropout(0.5),
            # second layer
            layers.Dense(width*4,  activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.5),
            # third layer
            layers.Dense(width,  activation='relu'), #512
            layers.BatchNormalization(),
            layers.Dropout(0.3),
    ])
    return model

In [17]:
# Create a MirroredStrategy
# strategy = tf.distribute.MirroredStrategy()

# Open a strategy scope
# with strategy.scope():

# Everything that creates variables should be under the strategy scope
model_supervised = tf.keras.models.Sequential([
    encoder(),
    tf.keras.layers.Dense(3, activation='softmax'),
])

model_supervised.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate= 1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False),
    metrics=['acc', tf.keras.metrics.AUC(name='auc')]
)

model_supervised.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_4 (Sequential)       │ (None, 128)            │    26,276,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,277,123 (100.24 MB)

 Trainable params: 26,220,675 (100.02 MB)

 Non-trainable params: 56,448 (220.50 KB)

In [18]:
history = model_supervised.fit(
    X_train,
    Y_train_cat,
    validation_data=(X_val, Y_val_cat),
    epochs=num_epochs,
    batch_size=batch_size,
    shuffle=True,
    verbose=1
)

Epoch 1/30
173/235 ━━━━━━━━━━━━━━━━━━━━ 8:00 8s/step - acc: 0.3312 - auc: 0.4965 - loss: 1.6781

KeyboardInterrupt: 